In [ ]:
import os
import copy
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:

# GPU CHECK


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# DATASET PATH


DATA_DIR = "Datasets/New-plant-diseases-dataset/"

train_dir = os.path.join(DATA_DIR, "train")
valid_dir = os.path.join(DATA_DIR, "valid")

In [ ]:
# TRANSFORMS


train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

valid_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

In [ ]:
# LOAD DATA


train_dataset = datasets.ImageFolder(train_dir, transform=train_transforms)
valid_dataset = datasets.ImageFolder(valid_dir, transform=valid_transforms)

class_names = train_dataset.classes
num_classes = len(class_names)

print("Total classes:", num_classes)
print("Classes:", class_names)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

In [ ]:
# MODEL: MOBILENETV2


model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)

for param in model.features.parameters():
    param.requires_grad = False

model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)

model = model.to(device)

In [ ]:
# LOSS, OPTIMIZER, SCHEDULER


criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(
    model.parameters(),
    lr=0.0001,
    weight_decay=0.0001
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    patience=2,
    factor=0.3
)

In [ ]:
# Mixed precision for faster GPU training
scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))

In [ ]:
# TRAINING FUNCTION

def train_model(model, epochs=20):
    best_model_weights = copy.deepcopy(model.state_dict())
    best_acc = 0.0
    patience = 5
    patience_counter = 0

    for epoch in range(epochs):
        start_time = time.time()

        print(f"\nEpoch {epoch + 1}/{epochs}")
        print("-" * 30)

        # Training
        model.train()
        train_loss = 0.0
        train_correct = 0

        for images, labels in train_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            optimizer.zero_grad()

            with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
                outputs = model(images)
                loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            _, preds = torch.max(outputs, 1)

            train_loss += loss.item() * images.size(0)
            train_correct += torch.sum(preds == labels.data)

        train_loss = train_loss / len(train_dataset)
        train_acc = train_correct.double() / len(train_dataset)

        # Validation
        model.eval()
        valid_loss = 0.0
        valid_correct = 0

        all_preds = []
        all_labels = []

        with torch.no_grad():
            for images, labels in valid_loader:
                images = images.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)

                with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
                    outputs = model(images)
                    loss = criterion(outputs, labels)

                _, preds = torch.max(outputs, 1)

                valid_loss += loss.item() * images.size(0)
                valid_correct += torch.sum(preds == labels.data)

                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        valid_loss = valid_loss / len(valid_dataset)
        valid_acc = valid_correct.double() / len(valid_dataset)

        scheduler.step(valid_acc)

        epoch_time = time.time() - start_time

        print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
        print(f"Valid Loss: {valid_loss:.4f} | Valid Acc: {valid_acc:.4f}")
        print(f"Time: {epoch_time:.2f} sec")

        # Save best model
        if valid_acc > best_acc:
            best_acc = valid_acc
            best_model_weights = copy.deepcopy(model.state_dict())
            torch.save({
                "model_state_dict": model.state_dict(),
                "class_names": class_names,
                "num_classes": num_classes
            }, "best_plant_disease_model.pth")

            print("Best model saved.")
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            print("Early stopping triggered.")
            break

    model.load_state_dict(best_model_weights)
    print(f"\nBest Validation Accuracy: {best_acc:.4f}")
    return model

In [ ]:
# START TRAINING


model = train_model(model, epochs=20)

print("Training completed.")

In [ ]:
from PIL import Image

def predict_image(image_path, model_path="best_plant_disease_model.pth"):
    checkpoint = torch.load(model_path, map_location=device)

    model = models.mobilenet_v2(weights=None)
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, checkpoint["num_classes"])
    model.load_state_dict(checkpoint["model_state_dict"])
    model = model.to(device)
    model.eval()

    class_names = checkpoint["class_names"]

    image = Image.open(image_path).convert("RGB")
    image = valid_transforms(image).unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(image)
        probabilities = torch.softmax(output, dim=1)
        confidence, predicted = torch.max(probabilities, 1)

    disease = class_names[predicted.item()]
    confidence = confidence.item() * 100

    return disease, confidence
